In [8]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [9]:
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)

In [10]:
# Convert SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)
print(f'spn datatype: {training_df['spn'].dtype}')

spn datatype: object


In [11]:
# features = [
#     'EventTimeStamp',
#     'EquipmentID',
#     'spn',
#     'fmi',
#     'active',
#     'Severity_Level',
#     'BarometricPressure',
#     'EngineCoolantTemperature',
#     'EngineLoad',
#     'EngineOilPressure',
#     'EngineOilTemperature',
#     'EngineRpm',
#     'FuelRate',
#     'FuelTemperature',
#     'IntakeManifoldTemperature',
#     'Speed',
#     'Throttle',
#     'TurboBoostPressure'
# ]
# len(features)

In [12]:
target = 'Derate_Target'

# Create dataset with features
X = training_df

# Create array of targets
y = training_df[target]

## Identify features for imputing missing values

In [13]:
# Group categorical columns
categorical_columns = ['EquipmentID', 'spn', 'fmi', 'active', 'Severity_Level']

# Group numeric columns
numeric_columns = [
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'Throttle',
    'TurboBoostPressure'
  ]

## Split training dataset

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

## Create pipeline and fit model

In [15]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# low_nan_numeric_pipe = Pipeline(
#     steps=[
#         ('scaler', StandardScaler()),
#         ('low_nan_numeric_imputer', SimpleImputer(strategy='median'))
#     ]
# )

numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('medium_nan_numeric_imputer', IterativeImputer(max_iter=20, random_state=30))
    ]
)

In [16]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        # ('low_nan_numeric_pipe', low_nan_numeric_pipe, low_nan_numeric_columns),
        ('medium_nan_numeric_pipe', numeric_pipe, numeric_columns)
    ]
)

In [17]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', MLPClassifier(
            activation='relu',
            hidden_layer_sizes=(32,32,32)
        ))
    ]
)

In [18]:
pipe.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EquipmentID', 'spn', 'fmi',
                                                   'active',
                                                   'Severity_Level']),
                                                 ('medium_nan_numeric_pipe',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('medium_nan_numeric_imputer',
                                                                   IterativeImputer(max_iter=20,
                                                                                    random_state=30))]),
                                                  ['BarometricPressure',
                                                   'EngineCoolantTemperature',
                                                   'EngineLoad',
                                                   'EngineOilPressure',
                                                   'EngineOilTemperature',
                                                   'EngineRpm', 'FuelRate',
                                                   'FuelTemperature',
                                                   'IntakeManifoldTemperature',
                                                   'Speed', 'Throttle',
                                                   'TurboBoostPressure'])])),
                ('model', MLPClassifier(hidden_layer_sizes=(32, 32, 32)))])

In [19]:
X_train['y_pred'] = pipe.predict(X_train)
X_test['y_pred'] = pipe.predict(X_test)

### Compare training and testing

In [20]:
labels = [0, 1, 2]

In [21]:
training_cr = classification_report(
    y_true=y_train,
    y_pred=X_train['y_pred'],
    digits=6
)
print(str(training_cr))

training_cm = confusion_matrix(
    y_true=y_train,
    y_pred=X_train['y_pred'],
    labels=labels
)
print(training_cm)

training_mcm = multilabel_confusion_matrix(
    y_true=y_train,
    y_pred=X_train['y_pred'],
    labels=labels
)
print(training_mcm)

              precision    recall  f1-score   support

           0   0.999235  0.999878  0.999556    739326
           1   0.770186  0.399356  0.525981       621
           2   0.879771  0.657632  0.752653       701

    accuracy                       0.999051    740648
   macro avg   0.883064  0.685622  0.759397    740648
weighted avg   0.998930  0.999051  0.998926    740648

[[739236     54     36]
 [   346    248     27]
 [   220     20    461]]
[[[   756    566]
  [    90 739236]]

 [[739953     74]
  [   373    248]]

 [[739884     63]
  [   240    461]]]


In [22]:
testing_cr = classification_report(
    y_true=y_test,
    y_pred=X_test['y_pred'],
    digits=6
)
print(str(testing_cr))

testing_cm = confusion_matrix(
    y_true=y_test,
    y_pred=X_test['y_pred']
)
print(testing_cm)

testing_mcm = multilabel_confusion_matrix(
    y_true=y_test,
    y_pred=X_test['y_pred'],
    labels=labels
)
print(testing_mcm)

              precision    recall  f1-score   support

           0   0.998902  0.999495  0.999199    316854
           1   0.268750  0.161049  0.201405       267
           2   0.607306  0.443333  0.512524       300

    accuracy                       0.998264    317421
   macro avg   0.624986  0.534626  0.571043    317421
weighted avg   0.997918  0.998264  0.998068    317421

[[316694     95     65]
 [   203     43     21]
 [   145     22    133]]
[[[   219    348]
  [   160 316694]]

 [[317037    117]
  [   224     43]]

 [[317035     86]
  [   167    133]]]
